<a href="https://colab.research.google.com/github/salty-arch/Flyrank-Intern/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

rel = "hf://datasets/FlyRank/internship-warehouse"

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one page's performance on one specific day,
for one client. The grain is `report_date × client_hash_id × content_hash_id`.

**Table used:** `fact_content_daily_performance`, partitioned by month.

**Time window:** my mid-panel month, `month=2026-03` (March 1–31, 2026).
I'm avoiding the `_sample` table (June 2026) for developing any label logic,
since that's the sealed final-month test partition.

Verified below: the grain check confirms no duplicate
(content, client, date) combinations, and the window check confirms the
full date span and row count for this partition.

In [10]:
grain_check = con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date, COUNT(*) AS c
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    GROUP BY content_hash_id, client_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""").df()
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [content_hash_id, client_hash_id, report_date, c]
Index: []


In [11]:
window_check = con.sql(f"""
    SELECT MIN(report_date) AS first_date, MAX(report_date) AS last_date, COUNT(*) AS total_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()
print(window_check)

  first_date  last_date  total_rows
0 2026-03-01 2026-03-31     9841378


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

**Context** (identify, join, or filter rows: never fed to the model as a signal):
- `report_date`, `client_hash_id`, `content_hash_id`, `month` — identifiers, used for grouping/joining
- `client_has_gsc`, `gsc_data_available` — gate whether a row's GSC numbers can be trusted
- `client_has_ga4`, `ga4_data_available` — same, for GA4 data (not used in this lane, kept for reference)

**Feature** (real inputs my scoring formula needs):
- `gsc_impressions` — visibility volume, used as the multiplier in my priority score
- `gsc_clicks` — used with impressions to compute CTR (`ctr = gsc_clicks / gsc_impressions`)
- `gsc_avg_position` — used to assign a page to a position tier, so I can compare its CTR
  against that tier's median CTR

**Label / proxy** (constructed, not a raw column):
- `priority_score = gap × gsc_impressions`, where `gap = tier_median_ctr - page_ctr`.
  This is a defined proxy I'm building, not an outcome directly observed in the data.

**Excluded** (with reason):
- `gsc_sum_position` — redundant given `gsc_avg_position` is already available
- `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec` —
  real signals, but outside this lane's specific question (search CTR vs. title, not on-site engagement)
- `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid`, `sessions_ai` —
  same reason, channel breakdown not needed for a title-rewrite priority question
- `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` —
  same reason, AI-referral breakdown not needed here
- `scroll_events` — same reason, on-page engagement not needed for this lane

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [12]:
grain_check = con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date, COUNT(*) AS c
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    GROUP BY content_hash_id, client_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""").df()
print("Grain check:")
print(grain_check)

window_check = con.sql(f"""
    SELECT MIN(report_date) AS first_date, MAX(report_date) AS last_date, COUNT(*) AS total_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()
print("\nWindow check:")
print(window_check)

availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()
print("\nAvailability check:")
print(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check:
Empty DataFrame
Columns: [content_hash_id, client_hash_id, report_date, c]
Index: []

Window check:
  first_date  last_date  total_rows
0 2026-03-01 2026-03-31     9841378


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Availability check:
   total_rows  available_rows
0     9841378         3611061


**What these three queries prove:**

1. **Grain check**: grouping by `content_hash_id`, `client_hash_id`, and
   `report_date` and filtering for count > 1 returns an empty result. This
   confirms my unit-of-analysis claim: no combination of page, client, and
   date ever repeats, so one row genuinely is one page's performance on one
   day, for one client.

2. **Row count + date span(Window check)**: this March partition (`month=2026-03`) spans
   the full month, 2026-03-01 through 2026-03-31, with 9,841,378 total rows.

3. **Availability check**: filtering to `gsc_data_available IS TRUE` keeps
   3,611,061 of 9,841,378 rows (~36.7%). This means roughly two-thirds of
   daily page-rows in this slice don't have trustworthy GSC data and must be
   excluded before any CTR or priority-score calculation. A real constraint
   on how much of the inventory I can actually score in a given month.

In [13]:
#Feature frame
feature_frame_w03 = con.sql(f"""
    WITH daily AS (
        SELECT
            content_hash_id,
            client_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            CASE WHEN report_date < DATE '2026-03-16' THEN 'first_half' ELSE 'second_half' END AS period
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available IS TRUE
    ),
    features AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions) AS feat_impressions,
            SUM(gsc_clicks) AS feat_clicks,
            AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS feat_avg_position,
            COUNT(*) AS feat_days_active,
            SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) AS feat_ctr
        FROM daily
        WHERE period = 'first_half'
        GROUP BY content_hash_id, client_hash_id
    ),
    label AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(CASE WHEN period = 'first_half' THEN gsc_impressions ELSE 0 END) AS first_half_impressions,
            SUM(CASE WHEN period = 'second_half' THEN gsc_impressions ELSE 0 END) AS second_half_impressions
        FROM daily
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        f.content_hash_id, f.client_hash_id,
        f.feat_impressions, f.feat_clicks, f.feat_avg_position, f.feat_days_active, f.feat_ctr,
        l.second_half_impressions,
        CASE
            WHEN (l.second_half_impressions - l.first_half_impressions) * 1.0 / NULLIF(l.first_half_impressions, 0) * 100 <= -20
            THEN TRUE ELSE FALSE
        END AS declining_flag
    FROM features f
    JOIN label l ON f.content_hash_id = l.content_hash_id AND f.client_hash_id = l.client_hash_id
    WHERE l.first_half_impressions > 0
""").df()

print(feature_frame_w03.shape)
feature_frame_w03.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(151981, 9)


,content_hash_id,client_hash_id,feat_impressions,feat_clicks,feat_avg_position,feat_days_active,feat_ctr,second_half_impressions,declining_flag
0,content_2e6360ad20fd7107,client_62f4a7e64f5e0096,219.0,1.0,4.004356,15,0.004566,680.0,False
1,content_4a1ca0fa5c177e0c,client_62f4a7e64f5e0096,11.0,0.0,5.238095,7,0.000000,3.0,True
2,content_c03ecafd4c999f15,client_62f4a7e64f5e0096,4974.0,9.0,8.155725,15,0.001809,5875.0,False
3,content_e689bc511192751a,client_62f4a7e64f5e0096,35.0,0.0,5.974359,15,0.000000,26.0,True
4,content_babcf791dccc1610,client_62f4a7e64f5e0096,101.0,0.0,7.767007,15,0.000000,80.0,True


**Feature notes (available at the decision moment because):**
- `feat_impressions`: knowable because it's summed only from March 1-15 (`WHERE period = 'first_half'`), before the moment being scored.
- `feat_clicks`: same window, same reason.
- `feat_avg_position`: averaged only across first-half days; zero-position rows (no real data) are excluded from the average.
- `feat_days_active`: a count of first-half days with any impressions, same window restriction.
- `feat_ctr`: computed from `feat_clicks / feat_impressions`, both first-half only.

In [14]:
#Honest baseline for trap comparison
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

clean_w03 = feature_frame_w03.dropna(subset=["feat_avg_position"])
honest_features = ["feat_impressions", "feat_clicks", "feat_avg_position", "feat_days_active", "feat_ctr"]

X = clean_w03[honest_features]
y = clean_w03["declining_flag"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model_honest = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
model_honest.fit(X_train, y_train)
honest_scores = model_honest.predict_proba(X_test)[:, 1]

print("Honest Precision@20:", precision_at_k(honest_scores, y_test.values, 20))

Honest Precision@20: 0.65


In [15]:
#Leakage trap
leaky_features = honest_features + ["second_half_impressions"]

X_leaky = clean_w03[leaky_features]

X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(
    X_leaky, y, test_size=0.2, random_state=42
)

model_leaky = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
model_leaky.fit(X_train_leaky, y_train_leaky)
leaky_scores = model_leaky.predict_proba(X_test_leaky)[:, 1]

print("Leaky Precision@20 (includes second_half_impressions):", precision_at_k(leaky_scores, y_test_leaky.values, 20))

Leaky Precision@20 (includes second_half_impressions): 1.0


**The leakage trap:**

I deliberately added `second_half_impressions`, a direct ingredient used to compute
`declining_flag`, as a sixth feature. Precision@20 jumped from 0.3 (honest, 5 features)
to 1.0 (leaky, 6 features). This is the expected signature of leakage: the model can
"look up" the answer rather than predict it, since since second_half_impressions directly hands the model information it's supposed to be predicting, rather than something it could genuinely know in advance. I removed this feature and
kept only the 0.3 score as my trusted, honest result.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What this data can never tell me:**

**Availability Gap**: The data can never tell me the true perfomance of every page because 63 % or almost 2/3rd of the page rows are filtered out because there is no GSC data available. This means I can only reliably score about a third of the inventory in a given month.

**Single-month Window**: Because I'm only looking at one month, this data cannot tell me about a pages activity on other months, it might just be that March was an unusually bad month for a given page, while earlier months looked fine.

**Uneven client history**: Clients joined at different times, so
   pages have different amounts of accumulated history behind them. A newer
   page might look like it's underperforming its position tier when really
   it just hasn't had enough time to build up impressions and clicks yet.
   Its low numbers could be a timing issue, not a real title problem.
   Comparing a brand-new page against a two-year-old page in the same tier
   isn't a perfectly fair comparison.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it. Done
- [x] The notebook runs top to bottom with no errors (Runtime → Run all). Done
- [x] No client names, URLs, or private queries anywhere. Done
- [x] My claims use careful words: observed, measured, directional, decision-support. Done
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.